In [ ]:
!pip install -q git+https://github.com/ficstamas/FateML.git

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

%matplotlib inline

# Dataset

This time for demonstrational purposes, we are going to rely on the [Fish Market Dataset](https://www.kaggle.com/aungpyaeap/fish-market). The dataset contains data about 159 fish, including their weight and dimensions. To understand what certain sizes mean, see the images below.

![Fish](https://github.com/ficstamas/FateML/raw/master/notebooks/images/fish_market_1.png)
![Fish](https://github.com/ficstamas/FateML/raw/master/notebooks/images/fish_market_2.png)

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/ficstamas/FateML/master/notebooks/data/fish_market.csv")
df

In [ ]:
df['Species'].unique()  # let's see the different species present in the dataset

In [ ]:
df.dtypes  
# This time pandas was able to guess the data types of each column correctly because we don't have missing values

In [ ]:
# this time we are going to make only the train and test split
train, test = train_test_split(df, train_size=0.7, random_state=0)

In [ ]:
train.describe()

In [ ]:
sns.pairplot(train, hue='Species')

# Regression vs Classification

**Regression and classification** are both types of supervised machine learning, but they are used for **different types of problems**.

**Regression** is used for **predicting a continuous value**, such as the price of a house or the temperature tomorrow. In a regression problem, the goal is to find a function that maps inputs to outputs, where the output is a continuous value.

**Classification**, on the other hand, is used for **predicting a discrete value**, such as whether an email is spam or not, or which type of animal is present on an image. In a classification problem, the goal is to find a function that maps inputs to discrete outputs, also called labels.

During **regression** our **target variable** is the **Weight**, and during **classification** our target variable is the **Species**.

| Target | Features |
| --- | --- |
| $y$ | $X$ |
| Dependent | Independent |
| Outcome | Design |
| Endogenous | Exogenous |

# Linear Regression

The basic idea behind linear regression is to find the line of best fit that minimizes the difference between the predicted values and the actual values. This line is represented by an equation of the form:

$$y^{(i)}=\beta_0+\sum_{j=1}^p\beta_jx_j^{(i)}=\beta_0+\overline{\beta}X$$

where $y^{(i)}$ is the dependent variable, $\beta_j$ is the slope of the line, $x_j^{(i)}$ is the independent variable, and $\beta_0$ is the intercept. $i$ represents a datapoint and $j$ is a feature.

It can be solved using closed form solution or optimization algorithms like gradient descent.
The closed form solution is a mathematical formula that gives the exact solution for the line of best fit, while optimization algorithms are iterative methods that find an approximate solution.

It's **important** to note that **linear regression** assumes a **linear relationship between the independent and dependent variables**, so it may not be appropriate for certain types of data.

In [ ]:
from fateml.data.fishmarket import prepare_for_regression

# splits has the following fields: train_x, train_y, dev_x, dev_y, test_x, test_y, other
# during these experiments we are going to use the train and test splits, dev is not even populated
# sklearn handles the intercept internally, so we don't need the statsmodels-format design matrix
splits = prepare_for_regression(df, standardize=False)

Linear Regression vizualized in 2D for with 1 feature. Assuming that we have 1 feature (Width) and pedicting the same target variable (Weight). In higher dimensions visualization becomes complicated.

In [ ]:
sns.jointplot(data=pd.concat([splits.train_x, splits.train_y], axis=1), x="Width", y="Weight", kind="reg")

In [ ]:
# model (sklearn)
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression(fit_intercept=True)
model = lr_model.fit(splits.train_x, splits.train_y)

y_pred = model.predict(splits.test_x)  # evaluating the model

Now that we have a trained model, we can investigate what the model has learned by analyzing the weights. Later we are going to have a look at the performance as well.

In [ ]:
import pandas as pd
import numpy as np

# numpy.r_ => Translates slice objects to concatenation along the first axis.

feature_names = list(splits.train_x.columns)
coefs = np.r_[model.intercept_, model.coef_.ravel()]  # intercept then coefficients
pd.Series(coefs, index=["Intercept"] + feature_names)

With a linear model, it is relatively easy to interpret the internal weights or parameters (also called coefficients or betas) because they have a direct relationship with the input features. In a linear model, the internal part of the summation ($\beta_jx_j^{(i)}$) represents a linear combination of the input features ($x_j^{(i)}$) and their corresponding weights or parameters ($\beta_j$). The weights can be thought of as the importance or contribution of each input feature to the final prediction. Positive weights indicate that the feature has a positive correlation with the target variable, while negative weights indicate a negative correlation.

So what happens when we increase the value of the `Height` (denote it as $h$) feature by 1?
$$\beta_hx_h^{(i)}\xrightarrow{+1}\beta_h(1+x_h^{(i)})=\beta_h + \beta_hx_h^{(i)}$$
If we increase the value by 1 then the prediction increases by $\beta_h$.

Let's see it in practice:

In [ ]:
datapoint = splits.test_x.iloc[:1].copy()
datapoint

What happens if we change a One-hot encoded category? 

For example, if a datapoint is one-hot encoded as a "Whitefish" but it is later identified as a "Pike".

`sklearn` models don't provide a full statistical summary table like `statsmodels`, but we can still inspect coefficients and compute common performance metrics (e.g., $R^2$, MSE) and even approximate t-statistics/p-values for OLS using the closed-form formulas.

In [ ]:
def model_statistics(x, y, model):
    # tstats
    y = y.values.squeeze()
    n, p = x.shape  # number of samples and features
    y_hat = model.predict(x).squeeze()  # predictions 
    beta_hat = np.r_[model.intercept_, model.coef_.ravel()]
    X_design = np.column_stack([np.ones(n), x.values])
    
    residuals = y - y_hat
    sse = np.sum(residuals**2)  # error term
    
    s2 = sse / (n - (p + 1))
    
    sigma = s2 * np.linalg.pinv(X_design.T @ X_design)  # covariance matrix
    SEs = np.sqrt(np.diag(sigma))  # standard error (SE)
    
    t_stats = beta_hat / SEs  # t- statistics
    
    
    try:
        from scipy import stats
        p_vals = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-(p+1)))
    except Exception:
        p_vals = np.full_like(t_stats, np.nan, dtype=float)
    
    feature_names = list(x.columns)
    coef_table = pd.DataFrame({
        "feature": ["Intercept"] + feature_names,
        "coef": beta_hat,
        "std err": SEs,
        "t": t_stats,
        "P>|t|": p_vals
    })
    return coef_table

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

# Basic fit diagnostics
y_true = splits.test_y.values.squeeze()
r2 = r2_score(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)

coef_table = model_statistics(splits.train_x, splits.train_y, model)

print("R2 (test)", r2)
print("MSE (test)", mse)
print("n (train)", splits.train_x.shape[0])
print("p (features)", splits.train_x.shape[1], "\n")
print(coef_table)

Model:
- $R^2$: The coefficient of determination, which is a measure of how well the model fits the data.

Features:
- **coef**: The coefficients (also known as weights or parameters) of the independent variables in the model.
- **std err**: The standard error of the coefficients, which is a measure of the uncertainty of the estimates.
- **t**: The t-statistic, which is the ratio of the coefficient to its standard error.
- $P>|t|$: The p-value, which is the probability of getting a t-statistic as extreme or more extreme than the observed value if the null hypothesis of no relationship between the independent variable and the dependent variable were true.
- \[0.025 , 0.975\]: The lower and upper bounds of the 95% confidence interval for the coefficient.

\*It's important to note that a normal distribution of residuals doesn't always imply a good fit, but it's a necessary condition for many statistical tests and assumptions to hold.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots()

x = np.arange(len(splits.test_x))

true_y = splits.test_y["Weight"].tolist()
pred_y = model.predict(splits.test_x).tolist()
ax.plot(x, true_y, "o", label="True")
ax.plot(x, pred_y, "or", label="LinearRegression prediction")
ax.vlines(x, true_y, pred_y, colors="gray", alpha=0.6)

ax.legend(loc="best")

In [ ]:
import pandas as pd
import numpy as np

pred = np.asarray(y_pred).ravel()
target = splits.test_y["Weight"].values if hasattr(splits.test_y, "__getitem__") and "Weight" in getattr(splits.test_y, "columns", []) else (splits.test_y.values.squeeze() if hasattr(splits.test_y,"values") else np.asarray(splits.test_y).squeeze())

conp_df = pd.DataFrame({"Prediction": pred, "Target": target})
sns.relplot(kind="scatter", data=conp_df, x="Prediction", y="Target")
plt.plot([pred.min(), pred.max()], [pred.min(), pred.max()], color="red")

## $R^2$ metric

The R-squared ($R^2$) metric is a statistical measure that represents the proportion of the variance in the dependent variable that is explained by the independent variables in a linear regression model. Ideally, it is a value between 0 and 1, with higher values indicating a better fit of the model to the data.

$R^2$ is calculated as the ratio of the explained variation to the total variation in the dependent variable. The explained variation is the variation of the predicted values (from the model) and the total variation is the variation of the actual values of the dependent variable.

In other words, $R^2$ measures the proportion of the variance in the dependent variable that can be explained by the independent variables and it's defined as:

$$R^2=1-\frac{SSE}{SST}$$
Sum of Squered Errors (SSE): 
$$SSE=\sum_{i=1}^n(y^{(i)}-\hat{y}^{(i)})^2,$$
$y^{(i)}$ dependent variable, and $\hat{y}^{(i)}$ is the prediction.
Sum of Squered Totals (SST):
$$SST=\sum_{i=1}^n(y^{(i)}-\overline{y}^{(i)})^2,$$
$\overline{y}^{(i)}$ denotes the expected value of the dependent variable of the data.

SSE expresses the variance of the model and SST defines the overall variance of the data.

Possible values:
- 0 $→$ couldn't fit a model at all
- 1 $→$ perfect fit
- negative value $→$ the model was not able to define the trend in the dataset

In [ ]:
true_values = splits.test_y.values.flatten()
predicted_values = y_pred.squeeze()



In [ ]:
from sklearn.metrics import r2_score

r2_score(splits.test_y, y_pred)

In [ ]:
r2_score([3, 2, 1], [2, 1, 1])

In [ ]:
r2_score([3, 2, 1], [1, 2, 3])

In [ ]:
#@title Generate SSE & SST plot
import matplotlib.pyplot as plt

xmin, xmax, ymin, ymax = -3, 3, -3, 3
ticks_frequency = 1

# vectors
xs, ys = [-3, -2, -1, 0, 1, 2, 3], [-2, -1, 0.5, 0.6, 1, 1.5, 2]
# Plot points
fig, axs = plt.subplots(1, 2, figsize=(8, 8))
# ax.scatter(xs, ys, c=colors)
ax = axs[0]
ax.set_title("SSE")
ax.scatter(xs, ys, c="blue")

ax.plot([-5, 5], [-5, 5], c="black", ls='-', lw=1.5, alpha=0.5)
ax.plot([-5, 5], [np.mean(ys), np.mean(ys)], c="orange", ls='-', lw=1.5, alpha=0.5)
# Draw lines connecting points to axes
for x, y in zip(xs, ys):
    ax.plot([x, x], [x, y], c="black", ls='--', lw=1.5, alpha=0.5)
    # ax.annotate(f"({x},{y})", (
    #     x + 0.03 if np.sign(x) == 1 else x - 0.13,
    #     y + 0.03 if np.sign(y) == 1 else y - 0.13
    # ), color="black", size=16)

# Set identical scales for both axes
ax.set(xlim=(xmin-1, xmax+1), ylim=(ymin-1, ymax+1), aspect='equal')

# Set bottom and left spines as x and y axes of coordinate system
ax.spines['bottom'].set_position('zero')
ax.spines['left'].set_position('zero')

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Create 'x' and 'y' labels placed at the end of the axes
# ax.set_xlabel('x', size=16, labelpad=-24, x=1.03)
# ax.set_ylabel('y', size=16, labelpad=-21, y=1.02, rotation=0)

# Create custom major ticks to determine position of tick labels
x_ticks = np.arange(xmin, xmax+1, ticks_frequency)
y_ticks = np.arange(ymin, ymax+1, ticks_frequency)
ax.set_xticks(x_ticks[x_ticks != 0])
ax.set_yticks(y_ticks[y_ticks != 0])

# Create minor ticks placed at each integer to enable drawing of minor grid
# lines: note that this has no effect in this example with ticks_frequency=1
ax.set_xticks(np.arange(xmin, xmax+1), minor=True)
ax.set_yticks(np.arange(ymin, ymax+1), minor=True)

# Draw major and minor grid lines
ax.grid(which='both', color='grey', linewidth=1, linestyle='-', alpha=0.2)

# Draw arrows
arrow_fmt = dict(markersize=4, color='black', clip_on=False)
ax.plot((1), (0), marker='>', transform=ax.get_yaxis_transform(), **arrow_fmt)
ax.plot((0), (1), marker='^', transform=ax.get_xaxis_transform(), **arrow_fmt)

# handles, labels = ax.get_legend_handles_labels()
# new_handles, new_labels, label_set = [], [], set()
# for h, l in zip(handles, labels):
#     if l in label_set:
#         continue
#     new_handles.append(h)
#     new_labels.append(l)
#     label_set.add(l)

# ax.legend(handles=new_handles, labels= new_labels, loc="lower left", prop={'size': 16})
ax = axs[1]

ax.set_title("SST")
ax.scatter(xs, ys, c="blue")

ax.plot([-5, 5], [-5, 5], c="black", ls='-', lw=1.5, alpha=0.5)
ax.plot([-5, 5], [np.mean(ys), np.mean(ys)], c="orange", ls='-', lw=1.5, alpha=0.5)
# Draw lines connecting points to axes
for x, y in zip(xs, ys):
    ax.plot([x, x], [np.mean(ys), y], c="orange", ls='--', lw=1.5, alpha=0.5)
    # ax.annotate(f"({x},{y})", (
    #     x + 0.03 if np.sign(x) == 1 else x - 0.13,
    #     y + 0.03 if np.sign(y) == 1 else y - 0.13
    # ), color="black", size=16)

# Set identical scales for both axes
ax.set(xlim=(xmin-1, xmax+1), ylim=(ymin-1, ymax+1), aspect='equal')

# Set bottom and left spines as x and y axes of coordinate system
ax.spines['bottom'].set_position('zero')
ax.spines['left'].set_position('zero')

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Create 'x' and 'y' labels placed at the end of the axes
# ax.set_xlabel('x', size=16, labelpad=-24, x=1.03)
# ax.set_ylabel('y', size=16, labelpad=-21, y=1.02, rotation=0)

# Create custom major ticks to determine position of tick labels
x_ticks = np.arange(xmin, xmax+1, ticks_frequency)
y_ticks = np.arange(ymin, ymax+1, ticks_frequency)
ax.set_xticks(x_ticks[x_ticks != 0])
ax.set_yticks(y_ticks[y_ticks != 0])

# Create minor ticks placed at each integer to enable drawing of minor grid
# lines: note that this has no effect in this example with ticks_frequency=1
ax.set_xticks(np.arange(xmin, xmax+1), minor=True)
ax.set_yticks(np.arange(ymin, ymax+1), minor=True)

# Draw major and minor grid lines
ax.grid(which='both', color='grey', linewidth=1, linestyle='-', alpha=0.2)

# Draw arrows
arrow_fmt = dict(markersize=4, color='black', clip_on=False)
ax.plot((1), (0), marker='>', transform=ax.get_yaxis_transform(), **arrow_fmt)
ax.plot((0), (1), marker='^', transform=ax.get_xaxis_transform(), **arrow_fmt)

However, regular R-squered score has its drawbacks:

1. The **regular R-squared score increases as the number of independent variables in the model increases**, even if those variables do not improve the model's ability to predict the dependent variable. The adjusted R-squared score accounts for this by adjusting the R-squared score based on the number of independent variables in the model.

2. The **regular R-squared score can be misleading when comparing models with different numbers of independent variables**. The adjusted R-squared score allows for a fair comparison of models with different numbers of independent variables.

$$\overline{R}^2=1-(1-R^2)\frac{n-1}{n-p-1},$$
$n$ denotes the number of datapoints and $p$ is the number of independent variables.

In conclusion, the adjusted R-squared score is a more reliable measure of the goodness of fit of a regression model and should be used when comparing models with different numbers of independent variables or when interpreting the overall predictive power of a model.

In [ ]:
from sklearn.metrics import r2_score

def r2_adj(true_vals, predictions, num_features):
    
    return 1

r2_adj(splits.test_y, y_pred, splits.train_x.shape[1])

## Feature importance

The importance of a feature can be defined as the absolute value of its t-statistics:
$$\mathcal{I}_{\hat{\beta}_j}=\left|\frac{\hat{\beta}_j}{SE(\hat{\beta}_j)}\right|,$$
$SE$ denotes the standard error.

We know that $\beta_j$ assigns a proportional contribution to our prediction which can be seen as the importance of that feature, and the standard error defines the uncertainity (variance) of that prediction. Let's see an example: if you need an immiadiate hearth transplant (high coefficient) but the only available person in the building is the janitor (low certainity or high variance), would you accept the outcome?

### Importance score

In [ ]:
statistics = model_statistics(splits.train_x, splits.train_y, model)


In [ ]:
g = sns.catplot(
    kind="bar",
    data=statistics,
    x="feature importance",
    y="feature",
    orient="h"
)

ax = g.ax  # get matplotlib axis

for bar, p in zip(ax.patches, statistics["P>|t|"]):
    if p > 0.05:
        bar.set_hatch("//")        # striped pattern
        bar.set_edgecolor("black")
        bar.set_alpha(0.5)         # fade non-significant bars

## Weight Plot

Weight plot visualizes the effect of each independent variable to the final contribution with a certain degree of variance.  The problem with the weight plot is that the features are measured on different scales. You can make it more comperable if you center and standardize the features before fitting the model.

In [ ]:
sns.pointplot(data=statistics[["feature", "coef"]], x="coef", y="feature", orient="h", linestyle='none', color='black')
plt.errorbar(y=np.arange(len(statistics)),x=statistics["coef"], xerr=statistics["std err"], fmt='none', c='black')
plt.axvline(x=0)

## Effect Plot

Effect of the model weights on the actual features:
$$effect^{(i)}_j=\beta_jx^{(i)}_j$$

In [ ]:
import pandas as pd
import numpy as np

feature_names = list(splits.test_x.columns)
coef_series = pd.Series(model.coef_.ravel(), index=feature_names)

effect = splits.test_x * coef_series
sns.catplot(kind="box", data=effect, orient="h")

### Boxplots Explained

Each boxplot consists of the following parts:
- Median ($Q2$/50th percentile): The middle value of the data set
- First Quartile ($Q1$/25th percentile): The middle number between the smallest number (not the “minimum”) and the median of the data set
- Third Quartile ($Q3$/75th percentile): The middle value between the median and the highest value (not the “maximum”) of the dataset
- Interquartile Range ($IQR$): 25th to the 75th percentile
- Whiskers (shown in blue)
- Outliers (shown as green circles)
- “Minimum”: $Q1 - 1.5*IQR$
- “Maximum”: $Q3 + 1.5*IQR$

[(Source)](https://builtin.com/data-science/boxplot)

![Boxplot](https://github.com/ficstamas/FateML/blob/master/notebooks/images/boxplots.jpg?raw=true)

In [ ]:
# dummy example
dist_x = [-5, -5, -3.7, -2, -1.1, 0.1, 0.5, 0.7, 1.1, 1.2, 105]

median = dist_x[int(len(dist_x) * 0.5)]
Q1 = dist_x[int(len(dist_x) * 0.25)]
Q3 = dist_x[int(len(dist_x) * 0.75)]
IQR = Q3 - Q1
minimum = Q1 - 1.5 * IQR
maximum = Q3 + 1.5 * IQR

minimum, Q1, median, Q3, maximum

# Lasso models

By increasing the number of independent variables (features) classical regression models become increasingly harder to interpretable. It is an ever occuring problem in real life scenarios. To alleviate this problem we are going to induce sparsity between the independent variables.

Lasso (sometimes also called as $\ell_1$ regularization) stands for “least absolute shrinkage and selection operator” which performs a feature selection inside the model. Classical linear optimization problem can be formulated as:

$$min_\beta\left(\frac{1}{n}\sum_{i=1}^n\left(y^{(y)}-\beta^tx\right)^2\right).$$

The $\ell_1$ regularization term is added to the cost function and it shrinks the less important feature's coefficient to zero thus, removing some features. This results in a sparse solution, where some of the feature coefficients are exactly equal to zero. This makes Lasso particularly useful for feature selection, as it automatically performs variable selection by setting some coefficients to zero. Thus our optimization problem looks like as:

$$min_\beta\left(\frac{1}{n}\sum_{i=1}^n\left(y^{(y)}-\beta^tx\right)^2+\lambda\|\beta\|_1\right),$$
where $\lambda$ controls the sparsity of the solution. As $\lambda$ increases so the sparsity in the model.


The main benefit of Lasso is that it helps to reduce overfitting. Overfitting occurs when a model is too complex, and it captures the noise in the data rather than the underlying trend. Lasso helps to overcome this problem by shrinking the coefficients of less important features to zero, effectively removing them from the model. Additionally, Lasso is also useful when you have a large number of correlated variables in your data, as it can select one variable from a group of highly correlated variables.

It is **important** to note that **Lasso is sensitive to the scale of the features**, therefore it is recommended to standardize the features before using them in a Lasso model.

![Fish](https://github.com/ficstamas/FateML/blob/e74a6fd43d820dce76f42d44750d87def88b383b/notebooks/images/regularization.png?raw=true)

In [ ]:
from fateml.data.fishmarket import prepare_for_regression

# we load the dataset again but standardize the parameters
splits = prepare_for_regression(df, standardize=True)

In [ ]:
from sklearn.linear_model import Lasso

# Note: sklearn's Lasso uses a different objective scaling than statsmodels' fit_regularized.
# Here we keep alpha=2.0 to match the notebook's hyperparameter value.
lasso = Lasso(alpha=2.0, fit_intercept=True, max_iter=10000)
model = lasso.fit(splits.train_x, splits.train_y.values.squeeze() if hasattr(splits.train_y, "values") else splits.train_y)

y_pred = model.predict(splits.test_x)

In [ ]:
from sklearn.metrics import r2_score

r2_score(splits.test_y, y_pred)

In [ ]:
r2_adj(splits.test_y, y_pred, splits.train_x.shape[1])

In [ ]:
statistics = model_statistics(splits.train_x, splits.train_y, lasso)
statistics["feature importance"] = statistics["t"].abs()

g = sns.catplot(
    kind="bar",
    data=statistics,
    x="feature importance",
    y="feature",
    orient="h"
)

ax = g.ax  # get matplotlib axis

for bar, p in zip(ax.patches, statistics["P>|t|"]):
    if p > 0.05:
        bar.set_hatch("//")        # striped pattern
        bar.set_edgecolor("black")
        bar.set_alpha(0.5)         # fade non-significant bars

In [ ]:
import pandas as pd
import numpy as np

feature_names = list(splits.test_x.columns)
coef_series = pd.Series(model.coef_.ravel(), index=feature_names)

effect = splits.test_x * coef_series
sns.catplot(kind="box", data=effect, orient="h")

# Logistic Regression

While linear regression cannot be used directly for classification, it can be adapted for classification by thresholding the predicted outcome. For example, if the predicted outcome is greater than a certain threshold, the model would predict one class, and if it is less than the threshold, it would predict the other class. The threshold is chosen based on the trade-off between sensitivity and specificity.

However, it would only interpolate the datapoints and it is not the best approach to solve the problem. Although we can fabricate a probabilistic distibution over our labels -- like in the example below -- it is highly sensitive for outliers as well. 

<img src='https://github.com/ficstamas/FateML/raw/master/notebooks/images/linear-class-threshold-1.png' width=500>

A better approach is to utilize a so called logit function:

<img src='https://github.com/ficstamas/FateML/raw/master/notebooks/images/logistic-class-threshold-1.png' width=500>

The logit function, also known as the logistic function, is a sigmoid function that maps any real-valued number to a value between 0 and 1. The logit function is defined as:
$$logit(x)=\frac{1}{1+e^{-x}}$$

Let's apply it on the output of the linear regression model which gives us the logistic regression model:
$$P(y^{(i)}=1)=\frac{1}{1+e^{-(\beta_0+\sum_{j=1}^p\beta_jx_j^{(i)})}}$$

In [ ]:
from fateml.data.fishmarket import prepare_for_classification
from fateml.data.utils import binarize_labels_in_splits

# we load the dataset again but standardize the parameters
splits = prepare_for_classification(df, standardize=True)
splits = binarize_labels_in_splits(splits, "Pike")  # We are going to predict whether it is a Pike or not

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000, solver="lbfgs")
model = logreg.fit(splits.train_x, splits.train_y.values.squeeze() if hasattr(splits.train_y, "values") else splits.train_y)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = splits.test_y.values.squeeze()
y_pred_cls = logreg.predict(splits.test_x)

print(classification_report(y_true, y_pred_cls))
confusion_matrix(y_true, y_pred_cls)

### Precision
**Definition:**  
The fraction of predicted positives that are actually correct.

**Formula:**
$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Interpretation:**  
When the model predicts a class, how often is it right?

---

### Recall
**Definition:**  
The fraction of actual positives that the model successfully finds.

**Formula:**
$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**Interpretation:**  
Of all true examples of a class, how many did the model detect?

---

### F1-score
**Definition:**  
The harmonic mean of precision and recall.

**Formula:**
$$
F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}
{\text{Precision} + \text{Recall}}
$$

**Interpretation:**  
A single metric balancing false positives and false negatives.

---

### Support
**Definition:**  
The number of true instances of a class in the dataset.

**Interpretation:**  
How many samples belong to that class.

---

### Accuracy
**Definition:**  
The fraction of total predictions that are correct.

**Formula:**
$$
\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Samples}}
$$

**Interpretation:**  
Overall percentage of correct classifications.

---

### Macro Average
**Definition:**  
The unweighted average of the metric across all classes.

**Interpretation:**  
Treats every class equally, regardless of size.

---

### Weighted Average
**Definition:**  
The average of the metric across classes weighted by support.

**Interpretation:**  
Large classes contribute more to the final score.

## Interpreting the weights

Due to the logit function each weight contributes to the final prediction in a non-linear way, thus we have to devise a new way to interpret them. First, let's consider the probability of an event occur:

$$\frac{P(y=1)}{1-P(y=1)}=\frac{P(y=1)}{P(y=0)}=odds$$ 

Output of the model is equal to:

$$logodds=\log\left(\frac{P(y=1)}{P(y=0)}\right)=\beta_0+\sum_{i=1}^p\beta_ix_i$$

Understanding a logarithm of ratios is still hard. However it is easy to convert $logodds$ to $odds$ by simply multiplying both sides with the inverze of logarithm:
$$\frac{P(y=1)}{1-P(y=1)}=odds=exp(\beta_0+\sum_{i=1}^p\beta_ix_i)$$

Now we can see how the outcome would change by increasing a dependent variable by 1. Although, we are going ot do it  in the ratio of odds. Let's say we increase $x_j$ by 1:
$$\frac{odds_{x_j+1}}{odds},$$
then expending the formula we get:
$$\frac{exp(\beta_0+\beta_1x_1+…+\beta_j(x_j+1)+…+\beta_px_p)}{exp(\beta_0+\beta_1x_1+…+\beta_jx_j+…+\beta_px_p)}.$$
We know that:
$$\frac{exp(a)}{exp(b)}=exp(a-b),$$
then applying it on the previous equation, the majority of the terms cancels out:
$$\frac{odds_{x_j+1}}{odds}=exp(\beta_j(x_j+1)-\beta_jx_j)=exp(\beta_j).$$

In conclusion, the odds ration would change with $exp(\beta_j)$ if we increase $x_j$ by 1. Obvously in general, if we change $x_j$ by $\Delta$:
$$\frac{odds_{x_j+1}}{odds}=exp(\beta_j(x_j+\Delta)-\beta_jx_j)=exp(\beta_j\Delta).$$

In [ ]:
import numpy as np
import pprint

delta = 10
sample_idx = 0
feature_name = 'Height'
datapoint = splits.test_x.iloc[sample_idx:sample_idx+1].copy()
loc_idx = datapoint.iloc[0].name
datapoint.loc[loc_idx, feature_name] = datapoint.loc[loc_idx, feature_name] + delta

feature_names = list(splits.train_x.columns)
height_idx = feature_names.index(feature_name)
beta_height = model.coef_.ravel()[height_idx]



# Practice

__Regression:__

- Load the Bike Rentail Dataset (a utility function is provided)
- Fit a linear regression model
- Visualize the importance, weight and effect plot
- Do you have any unimportant feature?
  - Are the importances are insignificant or can you drop them?
  - How does you model's performance (adjusted $R^2$) change if you drop the 3 most unimportant feature:
- How does standardization change the overall model and the easy of interpretation?
- Fit a Lasso model on the unchanged dataset:
  - Find a suitable $\lambda$ by using your development set.
  - In your final model, do you think the less importan features (according to the model) are the same as you dropped previously?

__Classification:__

- Load the Cervical Cancer dataset
- Fit a logistic regression model
  - What is your model's performance using a 0.5 threshold and measured by F1 metric?
  - Maximize the F1 score by selecting a suitable threshold (use your development set)
  - How would your odds change if we increase a person's age by 10?

In [ ]:
from fateml.data.bike_rental import load_dataset as load_dataset_bike

splits = load_dataset_bike(standardize=False)
splits

In [ ]:
from fateml.data.cervical_cancer import load_dataset as load_dataset_cancer

splits = load_dataset_cancer(standardize=True)
splits

# Further extensions of regression models

## Generalized Linear Models (GLM)

Generalized Linear Models (GLMs) are a generalization of traditional linear models that allow for the dependent variable to have a non-normal distribution. Some benefits of GLMs include:

1. Flexibility: GLMs can be used to model a wide range of dependent variables such as binary, count, and continuous data. This makes them more versatile than traditional linear models.

2. Robustness: GLMs are robust to outliers.

4. Link functions: GLMs allow the use of a link function to relate the linear predictor to the mean of the response variable. This allows for a wide range of distributions to be modeled. Such as the logistic function which models Bernoulli distribution. 

5. Model interpretability: GLMs provide clear and interpretable parameter estimates, which can be used to understand the relationship between the independent and the dependent variable.

6. Regularization: Some GLM models can also incorporate regularization techniques (such as Ridge and Lasso) that can help prevent overfitting and improve the generalization of the model.

In summary, GLMs provide a powerful framework for modeling a wide range of data types and distributions, while also providing interpretable parameter estimates and the ability to perform hypothesis testing and interval estimation.

Formalization of the model can be written as:

$$g(E_Y(y\mid x))=X^T\beta = \beta_0+\sum_{i=1}^p\beta_ix_i,$$
where $g$ is the link function and $E_Y$ is any distribution from the [exponential family](https://en.wikipedia.org/wiki/Exponential_family#Table_of_distributions).

In [ ]:
from fateml.data.fishmarket import prepare_for_regression

splits = prepare_for_regression(df, standardize=True)

In [ ]:
from sklearn.linear_model import PoissonRegressor

# Poisson regression (GLM) in sklearn
glm = PoissonRegressor(alpha=0.0, max_iter=1000)
model = glm.fit(splits.train_x, splits.train_y.values.squeeze() if hasattr(splits.train_y, "values") else splits.train_y)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_poisson_deviance

# sklearn-style inspection for PoissonRegressor
feature_names = list(splits.train_x.columns) if hasattr(splits.train_x, "columns") else [f"x{i}" for i in range(splits.train_x.shape[1])]
coef = np.r_[model.intercept_, model.coef_.ravel()]
pd.Series(coef, index=["Intercept"] + feature_names)

y_true = splits.test_y.values.squeeze() if hasattr(splits.test_y, "values") else np.asarray(splits.test_y).squeeze()
y_pred = model.predict(splits.test_x)
{"mean_poisson_deviance (test)": mean_poisson_deviance(y_true, y_pred)}

In [ ]:
from sklearn.metrics import r2_score

r2_score(splits.test_y, model.predict(splits.test_x))

In [ ]:
splits.test_x.columns

In [ ]:
from fateml.plots.poisson_regressor import plot_poisson_regressor_demo

results = plot_poisson_regressor_demo(
    df=pd.concat([splits.train_x, splits.train_y], axis="columns"),
    target="Weight",
    feature_x="Width",
    alphas=(0.0, 0.5, 1.0, 5.0),
)

## Generalized Additive Models (GAM)

Generalized Additive Models are attempting to solve the problem when the relationship between independent variables are non-linear. It is done by the inclusion of a non-linear function ($f_i$): 

$$\beta_0+\sum_{i=1}^p\beta_ix_i⇒\beta_0+\sum_{i=1}^pf_i(x_i)$$

It is important to note that, although GAMs are flexible and have benefits (such as handling of high-dimensional data, missing data or overdispersion), there are some cases where they may not perform as well as other models. For example, if the relationship between the predictors and the response variable is truly linear, a linear model may be a better choice.